**Amrit Dhillon GSB545 Homework #4 Ensemble + Feature Selection/Stacking Notebook**

**#1.** Build multiple models. The models should differ in a meaningful way, either in model type, feature representation, or tuning choices. This is important for stacking later. You are encouraged to explore models beyond those used heavily in class (e.g., Random Forest and XGBoost). Models should be tuned to improve performance. Improved predictions are of course one of the goals—it is a Kaggle competition—but a key goal of this assignment is to refine your personal workflow.

In [5]:
# imports
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import RandomizedSearchCV, cross_val_score
from sklearn.utils.class_weight import compute_sample_weight
import numpy as np
from sklearn.metrics import balanced_accuracy_score


In [4]:
train = pd.read_csv('/Users/amritdhillon/Desktop/Advanced ML/Kaggle Playground Series Competition/train.csv')
test = pd.read_csv('/Users/amritdhillon/Desktop/Advanced ML/Kaggle Playground Series Competition/test.csv')

#feature engineering (reusing what was used before)
for df in [train, test]:
    df['Moisture_to_Heat_Ratio'] = df['Soil_Moisture'] / (df['Temperature_C'] + 1)
    df['Water_Availability_Index'] = (df['Rainfall_mm'] + df['Previous_Irrigation_mm']) / (df['Field_Area_hectare'] + 1)
    df['Climate_Stress_Index'] = (df['Temperature_C'] * df['Sunlight_Hours'] * (1 + df['Wind_Speed_kmh'] / 25)) / (df['Humidity'] + 1)

    df['Temp_x_Humidity'] = df['Temperature_C'] * df['Humidity']
    df['Rainfall_x_SoilMoisture'] = df['Rainfall_mm'] * df['Soil_Moisture']
    df['Sunlight_x_Temp'] = df['Sunlight_Hours'] * df['Temperature_C']

X = train.drop(columns=['Irrigation_Need', 'id'])
y = train['Irrigation_Need']
X_test = test.drop(columns=['id'])

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# make categorical columns category dtype for XGBoost/LightGBM
cat_cols = X.select_dtypes(include='object').columns.tolist()

X_xgb = X.copy()
X_lgb = X.copy()

for col in cat_cols:
    X_xgb[col] = X_xgb[col].astype('category')
    X_lgb[col] = X_lgb[col].astype('category')

In [3]:
# XGBoost baseline + simple tuning sets
xgb_sets = [
    {"name": "Tuning Set 1", "params": {
        "random_state": 42,
        "enable_categorical": True,
        "eval_metric": "mlogloss",
        "tree_method": "hist"
    }},
    {"name": "Tuning Set 2", "params": {
        "random_state": 42,
        "enable_categorical": True,
        "eval_metric": "mlogloss",
        "tree_method": "hist",
        "learning_rate": 0.1,
        "max_depth": 6,
        "n_estimators": 200,
        "subsample": 1.0,
        "colsample_bytree": 0.8
    }},
    {"name": "Tuning Set 3", "params": {
        "random_state": 42,
        "enable_categorical": True,
        "eval_metric": "mlogloss",
        "tree_method": "hist",
        "learning_rate": 0.05,
        "max_depth": 8,
        "n_estimators": 300,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "min_child_weight": 5
    }}
]

xgb_results = []
for cfg in xgb_sets:
    model = XGBClassifier(**cfg["params"])
    scores = cross_val_score(model, X_xgb, y_encoded, cv=cv, scoring='balanced_accuracy', n_jobs=1)
    xgb_results.append({
        "model": cfg["name"],
        "cv_mean_bal_acc": scores.mean(),
        "cv_std": scores.std(),
        "fold_scores": [round(s, 6) for s in scores]
    })

xgb_results_df = pd.DataFrame(xgb_results).sort_values("cv_mean_bal_acc", ascending=False).reset_index(drop=True)
print("XGBoost results")
display(xgb_results_df)

XGBoost results


,model,cv_mean_bal_acc,cv_std,fold_scores
0,Tuning Set 2,0.962119,0.000463,"[0.961696, 0.962892, 0.96241, 0.961856, 0.961741]"
1,Tuning Set 3,0.962097,0.000749,"[0.960894, 0.963179, 0.962456, 0.961858, 0.962..."
2,Tuning Set 1,0.961684,0.000622,"[0.961166, 0.962781, 0.961911, 0.961493, 0.961..."


In [4]:
# LightGBM baseline + simple tuning sets
lgb_sets = [
    {"name": "Tuning Set 1", "params": {
        "objective": "multiclass",
        "random_state": 42,
        "verbose": -1
    }},
    {"name": "Tuning Set 2", "params": {
        "objective": "multiclass",
        "random_state": 42,
        "verbose": -1,
        "learning_rate": 0.03,
        "n_estimators": 500,
        "num_leaves": 63,
        "subsample": 0.8,
        "colsample_bytree": 0.8
    }},
    {"name": "Tuning Set 3", "params": {
        "objective": "multiclass",
        "random_state": 42,
        "verbose": -1,
        "learning_rate": 0.15,
        "n_estimators": 180,
        "num_leaves": 15,
        "max_depth": 6,
        "min_child_samples": 50,
        "reg_alpha": 0.5,
        "reg_lambda": 1.0
    }}
]

lgb_results = []
for cfg in lgb_sets:
    model = LGBMClassifier(**cfg["params"])
    scores = cross_val_score(model, X_lgb, y_encoded, cv=cv, scoring='balanced_accuracy', n_jobs=1)
    lgb_results.append({
        "model": cfg["name"],
        "cv_mean_bal_acc": scores.mean(),
        "cv_std": scores.std(),
        "fold_scores": [round(s, 6) for s in scores]
    })

lgb_results_df = pd.DataFrame(lgb_results).sort_values("cv_mean_bal_acc", ascending=False).reset_index(drop=True)
print("LightGBM results")
display(lgb_results_df)

LightGBM results


,model,cv_mean_bal_acc,cv_std,fold_scores
0,Tuning Set 2,0.962233,0.000685,"[0.961504, 0.962928, 0.962955, 0.961354, 0.962..."
1,Tuning Set 3,0.961666,0.000758,"[0.96103, 0.96264, 0.962378, 0.96066, 0.961622]"
2,Tuning Set 1,0.960603,0.001000,"[0.959502, 0.961832, 0.961759, 0.959724, 0.960..."


So after tuning three different sets of models for XGBoost and LightGBM I was able to improve my overall LightGBM model, but not my XGBoost model. I'm going to use the best version of each below before I move onto creating new features, tuning further, and moving onto other parts of the assignment.

- So I'm going to use the best parameters from the LightGBM model above using the RandomizedSearchCV from best XGBoost model

- I'm also going to use my best XGBoost model from the original homework assignment

In [7]:
#XGBoost best model (RandomizedSearchCV + sample weights)
#already have: X, y_encoded, cv, and feature engineering already applied to train/test above

X_xgb = X.copy()
cat_cols = X_xgb.select_dtypes(include='object').columns.tolist()
for col in cat_cols:
    X_xgb[col] = X_xgb[col].astype('category')

#class-imbalance handling
sample_w = compute_sample_weight(class_weight='balanced', y=y_encoded)

xgb_base = XGBClassifier(
    random_state=42,
    enable_categorical=True,
    eval_metric='mlogloss',
    tree_method='hist'
)

#best search space from before
xgb_param_dist = {
    "max_depth": [4, 6, 8, 10],
    "learning_rate": [0.03, 0.05, 0.1],
    "n_estimators": [100, 150, 200, 300],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0],
    "min_child_weight": [1, 3, 5]
}

xgb_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=xgb_param_dist,
    n_iter=12,
    scoring='balanced_accuracy',
    cv=cv,
    random_state=42,
    n_jobs=1,
    verbose=1
)

xgb_search.fit(X_xgb, y_encoded, sample_weight=sample_w)

print("Best XGBoost params:", xgb_search.best_params_)
print(f"Best XGBoost CV score: {xgb_search.best_score_:.6f}")


Fitting 5 folds for each of 12 candidates, totalling 60 fits
Best XGBoost params: {'subsample': 0.8, 'n_estimators': 150, 'min_child_weight': 5, 'max_depth': 6, 'learning_rate': 0.1, 'colsample_bytree': 0.8}
Best XGBoost CV score (search): 0.969511
Best XGBoost CV mean (recheck): 0.961723
Best XGBoost CV std  (recheck): 0.000704
Fold scores: [np.float64(0.960809), np.float64(0.962759), np.float64(0.962225), np.float64(0.961662), np.float64(0.961161)]


In [8]:
# LightGBM best-model search (RandomizedSearchCV + sample weights)

#already have: X, y_encoded, cv, and feature engineering already applied to train/test above

X_lgb = X.copy()
cat_cols = X_lgb.select_dtypes(include='object').columns.tolist()
for col in cat_cols:
    X_lgb[col] = X_lgb[col].astype('category')

sample_w = compute_sample_weight(class_weight='balanced', y=y_encoded)

lgb_base = LGBMClassifier(
    objective='multiclass',
    random_state=42,
    verbose=-1
)

#best search space so far
lgb_param_dist = {
    "learning_rate": [0.01, 0.03, 0.05, 0.1],
    "n_estimators": [300, 500, 700, 900],
    "num_leaves": [15, 31, 63, 127],
    "max_depth": [-1, 6, 10],
    "min_child_samples": [20, 40, 60],
    "subsample": [0.7, 0.8, 1.0],
    "colsample_bytree": [0.7, 0.8, 1.0],
    "reg_alpha": [0.0, 0.1, 0.5],
    "reg_lambda": [0.0, 0.5, 1.0, 2.0]
}

lgb_search = RandomizedSearchCV(
    estimator=lgb_base,
    param_distributions=lgb_param_dist,
    n_iter=12,
    scoring='balanced_accuracy',
    cv=cv,
    random_state=42,
    n_jobs=1,
    verbose=1
)

lgb_search.fit(X_lgb, y_encoded, sample_weight=sample_w)

print("Best LightGBM params:", lgb_search.best_params_)
print(f"Best LightGBM CV score (search): {lgb_search.best_score_:.6f}")

Fitting 5 folds for each of 12 candidates, totalling 60 fits
Best LightGBM params: {'subsample': 0.8, 'reg_lambda': 2.0, 'reg_alpha': 0.1, 'num_leaves': 15, 'n_estimators': 500, 'min_child_samples': 60, 'max_depth': 6, 'learning_rate': 0.05, 'colsample_bytree': 1.0}
Best LightGBM CV score (search): 0.970299


Well that took a while, but after waiting for my new best versions of XGBoost and LightGBM, I was able to:

- Tie my earlier best score for XGBoost

- Beat both my earlier LightGBM run with the new one, as well as my best XGBoost model as well

- My new best model is my current LightGBM model with a CV score of 0.970299

**#2.** Create and test new features. You should demonstrate reasonable effort in feature engineering, even if the features do not ultimately improve performance. Use a mix of approaches where appropriate (e.g., interactions, transformations, grouping, or other ideas). The goal is to explore different ways of representing the data, not just repeating one pattern.

In [9]:
for df in [train, test]:
    
    #Interactions
    df["Heat_Humidity_Stress"] = df["Temperature_C"] * df["Humidity"] #heat humidity interaction
    df["SoilRain_Interaction"] = df["Soil_Moisture"] * df["Rainfall_mm"] #soil moisture and rainfall interaction

    #Transformations
    df["Log_Rainfall"] = np.log1p(df["Rainfall_mm"]) #log transform to handle skewness in rainfall
    df["Temperature_Squared"] = df["Temperature_C"] ** 2 #capture non-linear temp effects

    #Grouping
    df["Temp_Band"] = pd.cut( #categorize temperature into bands
        df["Temperature_C"],
        bins=[-np.inf, 20, 30, np.inf],
        labels=["Cool", "Moderate", "Hot"]
    )

    df["Humidity_Band"] = pd.cut( #categorize humidity into bands
        df["Humidity"],
        bins=[-np.inf, 40, 70, np.inf],
        labels=["Low", "Medium", "High"]
    )

    df["Rainfall_Band"] = pd.cut( #categorize rainfall into bands
        df["Rainfall_mm"],
        bins=[-1e-9, 0, 20, np.inf],
        labels=["No_Rain", "Light_Rain", "Heavy_Rain"],
        include_lowest=True
    )

    #Other
    df["Rain_per_Area"] = df["Rainfall_mm"] / (df["Field_Area_hectare"] + 1) #rainfall per hectare to capture intensity of rain relative to field size


**#3.** Evaluate which features are useful. You may use feature importance, permutation importance, SHAP, LASSO, or other methods to support your decisions. The goal is to understand which features matter and why, not just to generate more features.

In [6]:
#XGBoost with best hyperparameters and sample weights (no tuning, evaluated on new feature set from features I made in previous cell)

X_xgb = X.copy()
cat_cols = X_xgb.select_dtypes(include='object').columns.tolist()
for col in cat_cols:
    X_xgb[col] = X_xgb[col].astype('category')

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

xgb_best = XGBClassifier(
    random_state=42,
    enable_categorical=True,
    eval_metric='mlogloss',
    tree_method='hist',
    subsample=0.8,
    n_estimators=150,
    min_child_weight=5,
    max_depth=6,
    learning_rate=0.1,
    colsample_bytree=0.8
)

xgb_fold_scores = []
for tr_idx, va_idx in cv.split(X_xgb, y_encoded):
    X_tr = X_xgb.iloc[tr_idx]
    X_va = X_xgb.iloc[va_idx]
    y_tr = y_encoded[tr_idx]
    y_va = y_encoded[va_idx]

    w_tr = compute_sample_weight(class_weight='balanced', y=y_tr)

    xgb_best.fit(X_tr, y_tr, sample_weight=w_tr)
    y_hat = xgb_best.predict(X_va)

    xgb_fold_scores.append(balanced_accuracy_score(y_va, y_hat))

print(f"XGBoost weighted CV balanced_accuracy mean: {np.mean(xgb_fold_scores):.6f}")
print(f"XGBoost weighted CV balanced_accuracy std:  {np.std(xgb_fold_scores):.6f}")
print("Fold scores:", [round(s, 6) for s in xgb_fold_scores])


XGBoost weighted CV balanced_accuracy mean: 0.969444
XGBoost weighted CV balanced_accuracy std:  0.000788
Fold scores: [0.968411, 0.969845, 0.970732, 0.969108, 0.969122]


In [7]:
#LightGBM with best hyperparameters and sample weights (no tuning, evaluated on new feature set from features I made in previous cell)

X_lgb = X.copy()
cat_cols = X_lgb.select_dtypes(include='object').columns.tolist()
for col in cat_cols:
    X_lgb[col] = X_lgb[col].astype('category')

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

lgb_best = LGBMClassifier(
    objective='multiclass',
    random_state=42,
    verbose=-1,
    subsample=0.8,
    reg_lambda=2.0,
    reg_alpha=0.1,
    num_leaves=15,
    n_estimators=500,
    min_child_samples=60,
    max_depth=6,
    learning_rate=0.05,
    colsample_bytree=1.0
)

lgb_fold_scores = []
for tr_idx, va_idx in cv.split(X_lgb, y_encoded):
    X_tr = X_lgb.iloc[tr_idx]
    X_va = X_lgb.iloc[va_idx]
    y_tr = y_encoded[tr_idx]
    y_va = y_encoded[va_idx]

    w_tr = compute_sample_weight(class_weight='balanced', y=y_tr)

    lgb_best.fit(X_tr, y_tr, sample_weight=w_tr)
    y_hat = lgb_best.predict(X_va)

    lgb_fold_scores.append(balanced_accuracy_score(y_va, y_hat))

print(f"LightGBM weighted CV balanced_accuracy mean: {np.mean(lgb_fold_scores):.6f}")
print(f"LightGBM weighted CV balanced_accuracy std:  {np.std(lgb_fold_scores):.6f}")
print("Fold scores:", [round(s, 6) for s in lgb_fold_scores])

LightGBM weighted CV balanced_accuracy mean: 0.970327
LightGBM weighted CV balanced_accuracy std:  0.000863
Fold scores: [0.969204, 0.970973, 0.971554, 0.969592, 0.970309]


- Within the cells above I took the best hyperparameters found for XGBoost and LightGBM and re-evaluated them using the new engineered features

- I used a 5-fold stratified CV in order to allow my models to use sample_weights which greatly helped my XGBoost models in earlier runs

- I got my best model yet in using the new feature set on the LightGBM best hyperparameter model, as well as getting a strong XGBoost model

- My current best model is LightGBM with a mean CV balanced accuracy of 0.970327

In [8]:
#fit both best models on full training data with sample weights and get feature importance

X_feat = X.copy()
cat_cols = X_feat.select_dtypes(include='object').columns.tolist()
for col in cat_cols:
    X_feat[col] = X_feat[col].astype('category')

w = compute_sample_weight(class_weight='balanced', y=y_encoded)

xgb_best = XGBClassifier(
    random_state=42,
    enable_categorical=True,
    eval_metric='mlogloss',
    tree_method='hist',
    subsample=0.8,
    n_estimators=150,
    min_child_weight=5,
    max_depth=6,
    learning_rate=0.1,
    colsample_bytree=0.8
)

lgb_best = LGBMClassifier(
    objective='multiclass',
    random_state=42,
    verbose=-1,
    subsample=0.8,
    reg_lambda=2.0,
    reg_alpha=0.1,
    num_leaves=15,
    n_estimators=500,
    min_child_samples=60,
    max_depth=6,
    learning_rate=0.05,
    colsample_bytree=1.0
)

xgb_best.fit(X_feat, y_encoded, sample_weight=w)
lgb_best.fit(X_feat, y_encoded, sample_weight=w)

xgb_imp = pd.Series(xgb_best.feature_importances_, index=X_feat.columns).sort_values(ascending=False)
lgb_imp = pd.Series(lgb_best.feature_importances_, index=X_feat.columns).sort_values(ascending=False)

print("Top 10 XGBoost features:")
display(xgb_imp.head(10).to_frame("xgb_importance"))

print("Top 10 LightGBM features:")
display(lgb_imp.head(10).to_frame("lgb_importance"))


Top 10 XGBoost features:


,xgb_importance
Crop_Growth_Stage,0.295422
Moisture_to_Heat_Ratio,0.170256
Soil_Moisture,0.137663
Mulching_Used,0.134318
Wind_Speed_kmh,0.065921
Temperature_C,0.061985
Rainfall_mm,0.040426
Rainfall_x_SoilMoisture,0.030914
Sunlight_x_Temp,0.011894
Temp_x_Humidity,0.005613


Top 10 LightGBM features:


,lgb_importance
Rainfall_mm,3050
Soil_Moisture,2393
Temperature_C,2035
Wind_Speed_kmh,1699
Crop_Growth_Stage,1327
Previous_Irrigation_mm,1186
Humidity,1112
Mulching_Used,1046
Rainfall_x_SoilMoisture,777
Sunlight_Hours,721


In [9]:
#comparing only engineered features between models

engineered_features = [
    "Moisture_to_Heat_Ratio",
    "Water_Availability_Index",
    "Climate_Stress_Index",
    "Temp_x_Humidity",
    "Rainfall_x_SoilMoisture",
    "Sunlight_x_Temp",
    "Heat_Humidity_Stress",
    "SoilRain_Interaction",
    "Log_Rainfall",
    "Temperature_Squared",
    "Temp_Band",
    "Humidity_Band",
    "Rainfall_Band",
    "Rain_per_Area"
]

feature_compare = pd.DataFrame({
    "feature": engineered_features
})

feature_compare["xgb_importance"] = feature_compare["feature"].map(xgb_imp).fillna(0)
feature_compare["lgb_importance"] = feature_compare["feature"].map(lgb_imp).fillna(0)

feature_compare = feature_compare.sort_values(
    ["lgb_importance", "xgb_importance"], ascending=False
).reset_index(drop=True)

feature_compare


,feature,xgb_importance,lgb_importance
0,Rainfall_x_SoilMoisture,0.030914,777.0
1,Moisture_to_Heat_Ratio,0.170256,642.0
2,Water_Availability_Index,0.003714,380.0
3,Temp_x_Humidity,0.005613,337.0
4,Climate_Stress_Index,0.003241,287.0
5,Sunlight_x_Temp,0.011894,273.0
6,Heat_Humidity_Stress,0.000000,0.0
7,SoilRain_Interaction,0.000000,0.0
8,Log_Rainfall,0.000000,0.0
9,Temperature_Squared,0.000000,0.0


*Overall*

- Both tuned models were driven most by core irrigation/climate variables, and they also gave meaningful weight to several of my engineered features

- Across both models, the strongest overall features were the non-engineered variables: Rainfall_mm, Soil_Moisture, Temperature_C, Wind_Speed_kmh, Crop_Growth_Stage, and Mulching_Used suggesting that irrigation need is driven primarily by direct moisture and weather conditions plus crop stage and field practice context

- Overall, feature importance shows that base environmental variables carry most predictive signal, while a subset of interaction and ratio engineered features provide incremental improvements

*Engineered Features*

- The strongest engineered features across both models were Moisture_to_Heat_Ratio and Rainfall_x_SoilMoisture (held importance in both models), so these were useful additions supporting the idea that moisture and weather conditions drive irrigation need most strongly

-  Water_Availability_Index, Temp_x_Humidity, Climate_Stress_Index, and Sunlight_x_Temp had smaller but still nonzero importance, so they contributed incrementally and are worth keeping for now

- Several new features had zero importance in both models, so they did not help and can be removed

**#4.** Combine your models using an ensemble method. This may include probability averaging (with equal or custom weights) or a meta-model (stacking). Evaluate whether the ensemble improves performance compared to individual models and support your evaluation with specific results.

In [10]:
train = pd.read_csv('/Users/amritdhillon/Desktop/Advanced ML/Kaggle Playground Series Competition/train.csv')
test = pd.read_csv('/Users/amritdhillon/Desktop/Advanced ML/Kaggle Playground Series Competition/test.csv')

#adjusted feature engineering
for df2 in [train, test]:
    df2['Moisture_to_Heat_Ratio'] = df2['Soil_Moisture'] / (df2['Temperature_C'] + 1)
    df2['Water_Availability_Index'] = (df2['Rainfall_mm'] + df2['Previous_Irrigation_mm']) / (df2['Field_Area_hectare'] + 1)
    df2['Climate_Stress_Index'] = (df2['Temperature_C'] * df2['Sunlight_Hours'] * (1 + df2['Wind_Speed_kmh'] / 25)) / (df2['Humidity'] + 1)
    df2['Temp_x_Humidity'] = df2['Temperature_C'] * df2['Humidity']
    df2['Rainfall_x_SoilMoisture'] = df2['Rainfall_mm'] * df2['Soil_Moisture']
    df2['Sunlight_x_Temp'] = df2['Sunlight_Hours'] * df2['Temperature_C']

    #features being removed
    #df['Heat_Humidity_Stress'] = df['Temperature_C'] * df['Humidity']
    #df['SoilRain_Interaction'] = df['Soil_Moisture'] * df['Rainfall_mm']
    #df['Log_Rainfall'] = np.log1p(df['Rainfall_mm'])
    #df['Temperature_Squared'] = df['Temperature_C'] ** 2
    #df['Temp_Band'] = pd.cut(df['Temperature_C'], bins=[-np.inf, 20, 30, np.inf], labels=['Cool', 'Moderate', 'Hot'])
    #df['Humidity_Band'] = pd.cut(df['Humidity'], bins=[-np.inf, 40, 70, np.inf], labels=['Low', 'Medium', 'High'])
    #df['Rainfall_Band'] = pd.cut(df['Rainfall_mm'], bins=[-1e-9, 0, 20, np.inf], labels=['No_Rain', 'Light_Rain', 'Heavy_Rain'], include_lowest=True)
    #df['Rain_per_Area'] = df['Rainfall_mm'] / (df['Field_Area_hectare'] + 1)

X = train.drop(columns=['Irrigation_Need', 'id'])
y = train['Irrigation_Need']
X_test = test.drop(columns=['id'])

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
n_classes = len(label_encoder.classes_)

#categorical dtype for tree models
cat_cols = X.select_dtypes(include='object').columns.tolist()
X_model = X.copy()
X_test_model = X_test.copy()

for col in cat_cols:
    X_model[col] = X_model[col].astype('category')
    all_cats = pd.Categorical(pd.concat([X_model[col], X_test_model[col]], axis=0)).categories
    X_model[col] = pd.Categorical(X_model[col], categories=all_cats)
    X_test_model[col] = pd.Categorical(X_test_model[col], categories=all_cats)


#best models from my tuning
xgb_best = XGBClassifier(
    random_state=42,
    enable_categorical=True,
    eval_metric='mlogloss',
    tree_method='hist',
    subsample=0.8,
    n_estimators=150,
    min_child_weight=5,
    max_depth=6,
    learning_rate=0.1,
    colsample_bytree=0.8
)

lgb_best = LGBMClassifier(
    objective='multiclass',
    random_state=42,
    verbose=-1,
    subsample=0.8,
    reg_lambda=2.0,
    reg_alpha=0.1,
    num_leaves=15,
    n_estimators=500,
    min_child_samples=60,
    max_depth=6,
    learning_rate=0.05,
    colsample_bytree=1.0
)

In [11]:
#OOF probabilities for fair ensemble evaluation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_xgb = np.zeros((len(X_model), n_classes))
oof_lgb = np.zeros((len(X_model), n_classes))

for tr_idx, va_idx in cv.split(X_model, y_encoded):
    X_tr, X_va = X_model.iloc[tr_idx], X_model.iloc[va_idx]
    y_tr, y_va = y_encoded[tr_idx], y_encoded[va_idx]

    w_tr = compute_sample_weight(class_weight='balanced', y=y_tr)

    xgb_best.fit(X_tr, y_tr, sample_weight=w_tr)
    lgb_best.fit(X_tr, y_tr, sample_weight=w_tr)

    oof_xgb[va_idx] = xgb_best.predict_proba(X_va)
    oof_lgb[va_idx] = lgb_best.predict_proba(X_va)

#individual model OOF scores
xgb_oof_pred = np.argmax(oof_xgb, axis=1)
lgb_oof_pred = np.argmax(oof_lgb, axis=1)

xgb_oof_score = balanced_accuracy_score(y_encoded, xgb_oof_pred)
lgb_oof_score = balanced_accuracy_score(y_encoded, lgb_oof_pred)

print(f"XGBoost OOF balanced_accuracy:  {xgb_oof_score:.6f}")
print(f"LightGBM OOF balanced_accuracy: {lgb_oof_score:.6f}")

#ensemble weight search
weight_rows = []
for w_lgb in np.arange(0.0, 1.01, 0.05):
    w_xgb = 1.0 - w_lgb
    p_mix = w_xgb * oof_xgb + w_lgb * oof_lgb
    y_mix = np.argmax(p_mix, axis=1)
    score = balanced_accuracy_score(y_encoded, y_mix)
    weight_rows.append({"w_xgb": w_xgb, "w_lgb": w_lgb, "oof_bal_acc": score})

weights_df = pd.DataFrame(weight_rows).sort_values("oof_bal_acc", ascending=False).reset_index(drop=True)
best_w_xgb = weights_df.loc[0, "w_xgb"]
best_w_lgb = weights_df.loc[0, "w_lgb"]
best_ens_oof = weights_df.loc[0, "oof_bal_acc"]

print(weights_df.head(10))
print(f"\nBest weights -> XGBoost: {best_w_xgb:.2f}, LightGBM: {best_w_lgb:.2f}")
print(f"Best Ensemble OOF balanced_accuracy: {best_ens_oof:.6f}")



XGBoost OOF balanced_accuracy:  0.969398
LightGBM OOF balanced_accuracy: 0.970271
   w_xgb  w_lgb  oof_bal_acc
0   0.10   0.90     0.970351
1   0.15   0.85     0.970339
2   0.30   0.70     0.970311
3   0.05   0.95     0.970310
4   0.20   0.80     0.970295
5   0.00   1.00     0.970271
6   0.40   0.60     0.970269
7   0.25   0.75     0.970262
8   0.45   0.55     0.970237
9   0.35   0.65     0.970231

Best weights -> XGBoost: 0.10, LightGBM: 0.90
Best Ensemble OOF balanced_accuracy: 0.970351


In [12]:
# summary table: single vs ensemble
summary_df = pd.DataFrame([
    {"model": "XGBoost", "oof_balanced_accuracy": xgb_oof_score},
    {"model": "LightGBM", "oof_balanced_accuracy": lgb_oof_score},
    {"model": f"Ensemble (XGB {best_w_xgb:.2f} + LGB {best_w_lgb:.2f})", "oof_balanced_accuracy": best_ens_oof}
]).sort_values("oof_balanced_accuracy", ascending=False).reset_index(drop=True)

summary_df


,model,oof_balanced_accuracy
0,Ensemble (XGB 0.10 + LGB 0.90),0.970351
1,LightGBM,0.970271
2,XGBoost,0.969398


In [13]:
#fit on full train for submission
w_full = compute_sample_weight(class_weight='balanced', y=y_encoded)

xgb_best.fit(X_model, y_encoded, sample_weight=w_full)
lgb_best.fit(X_model, y_encoded, sample_weight=w_full)

proba_xgb_test = xgb_best.predict_proba(X_test_model)
proba_lgb_test = lgb_best.predict_proba(X_test_model)

#best-weight submission
proba_best = best_w_xgb * proba_xgb_test + best_w_lgb * proba_lgb_test
pred_best = np.argmax(proba_best, axis=1)
label_best = label_encoder.inverse_transform(pred_best)

submission_best = pd.DataFrame({
    "id": test["id"],
    "Irrigation_Need": label_best
})

submission_best.to_csv(
    "/Users/amritdhillon/Desktop/Advanced ML/Kaggle Playground Series Competition/model outputs/submission_ensemble_weighted.csv",
    index=False
)
#got a new best kaggle score of 0.96842

In [15]:
row_count = len(submission_best);row_count

270000

In [16]:
#equal-weight submission in case of overfitting when I ran the best model submission
proba_equal = 0.5 * proba_xgb_test + 0.5 * proba_lgb_test
pred_equal = np.argmax(proba_equal, axis=1)
label_equal = label_encoder.inverse_transform(pred_equal)

submission_equal = pd.DataFrame({
    "id": test["id"],
    "Irrigation_Need": label_equal
})

submission_equal.to_csv(
    "/Users/amritdhillon/Desktop/Advanced ML/Kaggle Playground Series Competition/model outputs/submission_ensemble_equal.csv",
    index=False
)
#got even higher score on kaggle than the best weight submission, got a kaggle score of 0.96874

In [17]:
row_count = len(submission_equal);row_count

270000

In [18]:
# create Kaggle submission files for best single XGBoost and best single LightGBM from earlier to test against ensemble

w_full = compute_sample_weight(class_weight='balanced', y=y_encoded)

xgb_best.fit(X_model, y_encoded, sample_weight=w_full)
lgb_best.fit(X_model, y_encoded, sample_weight=w_full)

xgb_pred = np.argmax(xgb_best.predict_proba(X_test_model), axis=1)
lgb_pred = np.argmax(lgb_best.predict_proba(X_test_model), axis=1)

xgb_labels = label_encoder.inverse_transform(xgb_pred)
lgb_labels = label_encoder.inverse_transform(lgb_pred)

sub_xgb = pd.DataFrame({"id": test["id"], "Irrigation_Need": xgb_labels})
sub_lgb = pd.DataFrame({"id": test["id"], "Irrigation_Need": lgb_labels})

sub_xgb.to_csv(
    "/Users/amritdhillon/Desktop/Advanced ML/Kaggle Playground Series Competition/model outputs/submission_xgb_best_weighted.csv",
    index=False
)
#best model overall with kaggle submission score of 0.96905
sub_lgb.to_csv(
    "/Users/amritdhillon/Desktop/Advanced ML/Kaggle Playground Series Competition/model outputs/submission_lgb_best_weighted.csv",
    index=False
)
#improved over ensemble methods with kaggle submission score of 0.96878

In [21]:
row_count = len(sub_xgb);row_count
#row_count = len(sub_lgb);row_count

270000

In [22]:
compare_scores = pd.DataFrame([
    {"model": "XGBoost (best single model)", "kaggle_score": 0.96905},
    {"model": "LightGBM (best single model)", "kaggle_score": 0.96878},
    {"model": "Ensemble (0.9 - LightGBM / 0.1 - XGBoost)", "kaggle_score": 0.96842},
    {"model": "Ensemble (equal weights)", "kaggle_score": 0.96874},
])

compare_scores = compare_scores.sort_values("kaggle_score", ascending=False).reset_index(drop=True)
compare_scores["rank"] = compare_scores["kaggle_score"].rank(ascending=False, method="min").astype(int)

# gap from best leaderboard score in this table
best_score = compare_scores["kaggle_score"].max()
compare_scores["gap_from_best"] = compare_scores["kaggle_score"] - best_score

compare_scores

,model,kaggle_score,rank,gap_from_best
0,XGBoost (best single model),0.96905,1,0.00000
1,LightGBM (best single model),0.96878,2,-0.00027
2,Ensemble (equal weights),0.96874,3,-0.00031
3,Ensemble (0.9 - LightGBM / 0.1 - XGBoost),0.96842,4,-0.00063


- In my notebook OOF evaluation, the ensemble of XGBoost + LightGBM outperformed each single model, though this likely reflects some overfitting to cross-validation weighting/selection

- On Kaggle, both ensemble submissions improved over my earlier weeks’ results (equal = 0.96874, weighted = 0.96842), showing ensembling was helpful versus my past baselines

- After submitting updated single models with my best current feature set (removed redundant ones) + tuned hyperparameters, single-model performance was even better: XGBoost = 0.96905 and LightGBM = 0.96878

- Final ranking was XGBoost > LightGBM > equal-weight ensemble > weighted ensemble, and the margin from best ensemble to best XGBoost was not ver largee (0.00031), so improvements were incremental rather than dramatic between the models